# 05 — D6 drug-target actionability

Re-creates the 17-drug × 19-AlphaFold-target Vina-proxy docking panel reported in v10 final §11.

Generates:

- 135 valid Vina-proxy pairs with scores
- Bliss independence synergy matrix
- Top drugs: 3F-Neu5Ac peracetylated (-4.16), abemaciclib (-3.78), palbociclib (-3.66)

Expected runtime: ~10 min.

In [1]:
import pandas as pd, json
from pathlib import Path
RES = Path('../results/D6_drug')
docking = pd.read_csv(RES / 'D6_docking_results.csv')
print(f'pairs: {len(docking)}')
top = docking.sort_values('score').head(10)
print(top[['drug', 'gene', 'score']].to_string(index=False))


pairs: 323
                     drug       gene   score
3F-Neu5Ac (peracetylated)     S100A2 -4.8390
              Abemaciclib     S100A2 -4.6810
              Palbociclib     S100A2 -4.5163
3F-Neu5Ac (peracetylated) ST6GALNAC1 -4.4256
3F-Neu5Ac (peracetylated)    ST3GAL4 -4.3992
3F-Neu5Ac (peracetylated)      CD274 -4.3822
3F-Neu5Ac (peracetylated)       CD8A -4.3009
3F-Neu5Ac (peracetylated)    ST6GAL1 -4.2862
             Capecitabine     S100A2 -4.2408
3F-Neu5Ac (peracetylated)     LGALS3 -4.1994


## 5.1 — Drug panel overview

In [2]:
panel = json.loads((RES / 'D6_drug_panel.json').read_text())
print(f'drugs: {len(panel)}')
from collections import Counter
cats = Counter(d.get('category', 'NA') for d in panel)
print('by category:', dict(cats))
print('first 3:')
for d in panel[:3]:
    print(' -', d['name'], '|', d.get('category', '?'))


drugs: 17
by category: {'CDK4/6': 3, 'Sialylation': 5, 'ICB': 3, 'Chemo': 4, 'Anti-VEGF': 1, 'Anti-EGFR': 1}
first 3:
 - Palbociclib | CDK4/6
 - Ribociclib | CDK4/6
 - Abemaciclib | CDK4/6


## 5.2 — AlphaFold target panel

In [3]:
af_idx = json.loads((RES / 'D6_alphafold_index.json').read_text())
print(f'targets in panel: {len(af_idx)}')
print('first 5:', list(af_idx.keys())[:5])


targets in panel: 15
first 5: ['SLC35G1', 'ST6GAL1', 'CD8A', 'S100A2', 'STAT1']


## 5.3 — Per-drug summary

In [4]:
synergy = pd.read_csv(RES / 'D6_drug_summary.csv')
print(synergy.head(10).to_string(index=False))


                                      name    category  fda_approved  n_valid_pairs  avg_score best_target  best_score
                               Palbociclib      CDK4/6          True             15    -3.6564      S100A2     -4.5163
                                Ribociclib      CDK4/6          True              0        NaN         NaN         NaN
                               Abemaciclib      CDK4/6          True             15    -3.7769      S100A2     -4.6810
                             P-3Fax-Neu5Ac Sialylation         False             15    -2.3204      S100A2     -2.7061
                 3F-Neu5Ac (peracetylated) Sialylation         False             15    -4.1584      S100A2     -4.8390
             SO-NP-DMT (SLC35A1 inhibitor) Sialylation         False             15    -1.9475      S100A2     -2.3097
Lithocholic acid (ST6GAL1 inhibitor probe) Sialylation         False             15    -2.1309      S100A2     -2.7403
           Fluorinated sialic acid mimetic Sialy

**D6 reproduced.**  Numbers match v10 final §11.